In [1]:
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# --- standard library -----------------------------------------------------
import json
import logging
from collections import Counter

# --- third party ----------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- project --------------------------------------------------------------
import config
from src.data import fetch_poems, generate, splits
from src.data import filter as data_filter  # `filter` alone shadows the builtin
from src.plots import figures
from src.eval import format_check, grounding, judge, swap_test

%load_ext autoreload
%autoreload 2

# One logging setup for every notebook. Also quietens the HTTP and hub loggers
# that would otherwise bury this project's own output — see config for why each
# suppression is safe.
config.configure_logging()

# 3 — Judge validation

**This notebook is a gate, and it runs before any GPU time is spent.**

Everything downstream rests on one assumption: that an LLM judge can tell an
interpretation written *for* a poem from one written for a different poem. If
it cannot, then every score it produces for the five arms is noise wearing a
number, and no amount of training or evaluation repairs that.

So the assumption is tested first, on the easiest possible case — **teacher
interpretations**, which the funnel has already verified quote their poem
correctly. These are known-grounded text. A judge that cannot separate matched
from mismatched *here* has no chance on a 0.5B model's output.

The design is the swap test. One interpretation, three poems:

| Condition | Poem shown | Role |
|---|---|---|
| `matched` | the poem it was written for | — |
| `mismatched_random` | a random poem by a **different** author | standard control |
| `mismatched_same_author` | a different poem by the **same** author | strict control |

Two gaps follow, and their difference is the finding:

- **grounding gap** = matched − mismatched_random
- **poem-level gap** = matched − mismatched_same_author
- **author component** = the difference between them

The third row is why the strict control exists. An author's themes recur, so
text that merely sounds like Dickinson will beat a random Whitman poem while
scoring no better on one Dickinson poem than another. The standard control
alone would call that well grounded.

**This is a relative measurement, which is why it needs no ground truth.** The
judge's scale is never calibrated — only kept consistent. Miscalibration moves
every condition together and cancels in the difference.

## Building the pairs

Every constraint below fails *silently*. Nothing raises if the strict control
draws a near-duplicate, or if the standard control happens to draw the same
author — the run completes and reports a plausible number that means something
else. `swap_test.check_all` is what turns those into errors instead of
findings.

In [ ]:
corpus, _ = data_filter.build_corpus(fetch_poems.load_cached(),
                                    generate.load_cached())

# PoetryDB publishes some poems under several titles. Drawing one as the
# "different poem by the same author" would turn the strict condition into the
# matched condition, and the poem-level gap would collapse to zero for a reason
# that has nothing to do with the model.
duplicates = data_filter.near_duplicate_ids(corpus)

# The same 150 evaluation poems the arms will later be generated for, drawn
# with the same seed — so this validation and the real run judge the same poems.
exemplars = splits.reserve_exemplars(corpus, n=config.N_FEWSHOT, seed=config.SEED)
folds = splits.make_folds(corpus, config.N_FOLDS,
                          group_key=config.FOLD_GROUP_KEY,
                          seed=config.SEED, exclude=exemplars)
eval_set = splits.sample_eval_poems(folds, config.EVAL_PER_FOLD, config.SEED)

pairs = swap_test.build_pairs(
    [{"poem_id": p["poem_id"], "interpretation": p["interpretation"]}
     for p in eval_set],
    corpus, arm="teacher", near_duplicates=duplicates)

swap_test.check_all(pairs, corpus, near_duplicates=duplicates)
print(f"{len(pairs)} pairs — {len(pairs) // len(config.SWAP_CONDITIONS)} "
      f"interpretations x {len(config.SWAP_CONDITIONS)} conditions")
print(f"{len(pairs) * len(config.JUDGES)} calls if both judges score everything")

## One pair, read by eye first

Before trusting 450 automated scores, read one. The three rows below are the
same interpretation judged against three different poems — if the design is
sound, the first should score high and the other two low.

In [ ]:
by_id = {p["poem_id"]: p for p in corpus}
example = [p for p in pairs if p.poem_id == pairs[0].poem_id]

print(f'INTERPRETATION written for: "{by_id[example[0].poem_id]["title"]}" '
      f'by {by_id[example[0].poem_id]["author"]}\n')
print(example[0].interpretation[:400], "...\n")
print("scored against:")
for pair in example:
    shown = by_id[pair.shown_id]
    print(f'  {pair.condition:<24} "{shown["title"][:34]}" by {shown["author"]}')

## Scoring

**Resumable and cached.** Every score is appended to
`results/judge_scores_<judge>.jsonl` the moment it arrives, and already-scored
pairs are skipped on restart — so an interrupted run loses nothing and
re-running this cell costs nothing. That file is committed with the repo, which
is how these results survive the session.

Scores are keyed by judge name in the **filename as well as in every record**,
so the two judges cannot be pooled even by accident. `assert_single_judge`
raises on a mixed set rather than quietly averaging across two instruments.

In [ ]:
scored = {}
for spec in config.JUDGES:
    try:
        scored[spec.name] = judge.score_all(pairs, corpus, spec)
    except Exception as error:
        print(f"{spec.name}: {type(error).__name__}: {str(error)[:200]}")

for name, records in scored.items():
    usable = [r for r in records if r.get("score") is not None]
    print(f"{name:<14} {len(usable)}/{len(pairs)} scored, "
          f"{len(records) - len(usable)} unparseable")

## The gate

`MIN_JUDGE_SEPARATION` is stated as an effect size, not a p-value. With 150
paired observations almost any non-zero gap is statistically significant; the
question here is whether the instrument discriminates *usefully*, and one point
on a ten-point scale is the least that could be called separation.

**A failure here is not a bad result — it is a stop sign.** It would mean the
judge cannot tell grounded from ungrounded on text known to be grounded, and
the design has to change before any GPU time is spent.

In [ ]:
for name, records in scored.items():
    if not records:
        continue
    print(f"--- {name} ---")
    for condition, mean in judge.condition_means(records).items():
        print(f"  {condition:<24} {mean:.2f}")
    gaps = judge.gaps(records)
    print(f"  {'grounding gap':<24} {gaps['grounding_gap']:+.2f}   "
          f"(matched - random)")
    print(f"  {'poem-level gap':<24} {gaps['poem_level_gap']:+.2f}   "
          f"(matched - same author)")
    print(f"  {'author component':<24} {gaps['author_component']:+.2f}   "
          f"(the difference)")
    passed, verdict = judge.separation_verdict(records)
    print(f"\n  {verdict}\n")

## Saving the result

Written to `results/swap_test_summary.csv`, one **row per judge** — never a
pooled mean, which would answer a question nobody asked. Generated rather than
hand-typed, so the numbers in the report cannot drift from the scores they came
from.

Two artifacts persist beyond this session, both committed:

| File | What it holds |
|---|---|
| `results/judge_scores_<judge>.jsonl` | every individual score, resumable |
| `results/swap_test_summary.csv` | the aggregate, one row per judge |

In [ ]:
summary = judge.save_summary([r for r in scored.values() if r])
summary

## What this does and does not establish

**Establishes:** whether each judge separates matched from mismatched on
known-grounded text, and how much of that separation survives the strict
same-author control.

**Does not establish:** that the judge agrees with human readers. Zheng et al.
2023 report >80% judge–human agreement on MT-Bench, but that is their benchmark
and their task; no human annotation exists for poetry grounding here, and none
is claimed. What is measured is agreement with a *known-correct pairing*, which
is a weaker but genuinely different check.

**The author component is the number to carry forward.** If it is large, then
much of what looks like grounding is the judge recognising an author rather
than reading a poem — and the poem-level gap, not the grounding gap, is the
defensible measurement for every arm that follows.